# Leaf Wood Segmentation

build on top of https://github.com/qforestlab/leaf-wood-segmentation-with-deep-learning to perform the segmentation

In [1]:
%load_ext autoreload
%autoreload 2

## Download weights

In [2]:
from pyprojroot import here
import requests
from pathlib import Path
import zipfile
import laspy
import numpy as np

In [3]:
url = "https://zenodo.org/records/13767795/files/model_weights.zip?download=1"
weights_dir = here("data/weights")
weights_dir.mkdir(parents=True, exist_ok=True)
dst = weights_dir / "model_weights.zip"

In [5]:
if not dst.exists():
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

with zipfile.ZipFile(dst, "r") as z:
    z.extractall(weights_dir)

In [4]:
def read_cloud(path):
    pcl = laspy.read(path).xyz
    return {
        'point': pcl,
        'feat': None,
        'label': np.zeros((pcl.shape[0]), dtype=np.int32)
    }

## Open3d Model

In [5]:
import sys
sys.prefix

'/Stor3/simone/semantic-segmentation/.pixi/envs/default'

In [6]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import open3d.ml.torch as ml3d
from open3d.ml.torch.datasets import Custom3D
from open3d.ml.torch.modules import losses
from open3d.ml.utils import Config

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [7]:
class CustomPointTransformer(ml3d.models.PointTransformer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        num_per_class = self.cfg.get('class_weights', None)
        self.class_weights = self.get_class_weights(num_per_class) if num_per_class is not None else None
        self.cce_loss = nn.CrossEntropyLoss(weight=self.class_weights, reduction='none')

    def get_class_weights(self, num_per_class):
        num_per_class = np.array(num_per_class, dtype=np.float32)
        weight = num_per_class / float(sum(num_per_class))
        ce_label_weight = 1 / (weight + 0.02)
        return torch.tensor(ce_label_weight)
    
    def get_loss(self, Loss, results, inputs, device):
        labels = inputs['data'].label

        scores, labels = losses.filter_valid_label(
            results, 
            labels, 
            self.cfg.num_classes,
            self.cfg.ignored_label_inds,
            device,
        )
        loss = self.cce_loss(scores, labels).mean()
        return loss, labels, scores

In [8]:
torch.cuda.is_available()

True

In [9]:
model_config = {'name': 'PointTransformer',
 'batcher': 'ConcatBatcher',
 'ckpt_path': str(here("data/weights/model_weights/weights_pointtransformer.pth")),
 'is_resume': False,
 'in_channels': 3,
 'blocks': [2, 3, 4, 6, 3],
 'num_classes': 2,
 'voxel_size': 0.02,
 'max_voxels': 65536,
 'ignored_label_inds': [],
}

In [10]:
pipeline_config = {
'name': 'SemanticSegmentation',
#  'device': 'cuda:0',
'device': 'cpu',
 'num_workers': 16,
 'batch_size': 1,
}

In [11]:
cfg = Config.load_from_file(str(here("nbs/model_pipeline.yml")))

In [12]:
model = CustomPointTransformer(**model_config)
pipeline = ml3d.pipelines.SemanticSegmentation(model, **pipeline_config)

In [13]:
# model = CustomPointTransformer(**cfg.model)
# pipeline = ml3d.pipelines.SemanticSegmentation(model, **cfg.pipeline)

In [14]:
pipeline.load_ckpt(here("data/weights/model_weights/weights_pointtransformer.pth"))

## Run

In [15]:
tile0 = read_cloud(here("data/1_tiled/tile_1.laz"))

In [23]:
def read_cloud_old(path):
    if path[-3:] == 'npy':
        pcl = np.load(path)
    elif path[-3:] == 'txt':
        pcl = np.loadtxt(path)
    elif path[-3:] == 'ply':
        pcl = o3d.t.io.read_point_cloud(path)
        pcl = pcl.point.positions.numpy()
    else:
        raise ValueError('input file should be txt, ply or npy')
    
    data = {
        'point': pcl[:, :3],
        'feat': np.zeros((pcl.shape[0], 1), dtype=np.float32),
        'label': np.zeros((pcl.shape[0]), dtype=np.int32)
    }
    return data

In [24]:
tile_old = read_cloud_old(str(here("data/0_raw/tls_tropical_leaf_wood/test/dro_033_pc.txt")))
tile_old

{'point': array([[-4.032, 92.813, -6.123],
        [-3.841, 92.603, -7.013],
        [-3.842, 92.605, -6.992],
        ...,
        [-1.856, 96.338, 11.659],
        [-1.916, 96.421, 11.704],
        [-1.841, 96.335, 11.697]]),
 'feat': array([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]], dtype=float32),
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32)}

In [29]:
pipeline.run_inference(tile_old)

[0]
now trying <class 'numpy.ndarray'>


RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [29]:
tile0

{'point': array([[3.38458476e+05, 8.07359535e+06, 1.07546150e+03],
        [3.38458723e+05, 8.07359534e+06, 1.07543275e+03],
        [3.38458736e+05, 8.07359535e+06, 1.07543775e+03],
        ...,
        [3.38469316e+05, 8.07360222e+06, 1.10911850e+03],
        [3.38469325e+05, 8.07360224e+06, 1.10913000e+03],
        [3.38469088e+05, 8.07360276e+06, 1.10921125e+03]]),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32)}

In [30]:
np.__version__

'1.26.4'

In [31]:
model

CustomPointTransformer(
  (encoders): ModuleList(
    (0): Sequential(
      (0): TransitionDown(
        (linear): Linear(in_features=3, out_features=32, bias=False)
        (bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (1): Bottleneck(
        (linear1): Linear(in_features=32, out_features=32, bias=False)
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (transformer2): Transformer(
          (linear_q): Linear(in_features=32, out_features=32, bias=True)
          (linear_k): Linear(in_features=32, out_features=32, bias=True)
          (linear_v): Linear(in_features=32, out_features=32, bias=True)
          (linear_p): Sequential(
            (0): Linear(in_features=3, out_features=3, bias=True)
            (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Li

In [32]:
import open3d as o3d

In [33]:
model.device = 'cuda:0'

In [34]:
model.eval()

CustomPointTransformer(
  (encoders): ModuleList(
    (0): Sequential(
      (0): TransitionDown(
        (linear): Linear(in_features=3, out_features=32, bias=False)
        (bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (1): Bottleneck(
        (linear1): Linear(in_features=32, out_features=32, bias=False)
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (transformer2): Transformer(
          (linear_q): Linear(in_features=32, out_features=32, bias=True)
          (linear_k): Linear(in_features=32, out_features=32, bias=True)
          (linear_v): Linear(in_features=32, out_features=32, bias=True)
          (linear_p): Sequential(
            (0): Linear(in_features=3, out_features=3, bias=True)
            (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Li

In [35]:
preprocess_func = model.preprocess
processed_data = preprocess_func(tile_old, {'split': 'test'})

In [36]:
processed_data

{'point': array([[ 4.123    ,  4.5      , 20.796    ],
        [ 4.142    ,  4.4940033, 20.726    ],
        [ 4.0680003,  4.5350037, 20.711    ],
        ...,
        [ 2.0490003,  1.9360046, 19.813    ],
        [ 3.2040002,  3.459999 , 19.11     ],
        [ 3.1220002,  3.0270004, 19.178001 ]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32),
 'search_tree': <sklearn.neighbors._kd_tree.KDTree at 0x5eae405923d0>,
 'proj_inds': array([47971, 43877, 36730, ...,  7020, 53686,     0], dtype=int32)}

In [37]:
from open3d.ml.torch.dataloaders import TorchDataloader, DefaultBatcher, ConcatBatcher, get_sampler
from open3d.ml.torch.datasets import InferenceDummySplit

In [41]:
from torch.utils.data import DataLoader

In [52]:
infer_sampler = infer_dataset.sampler

In [53]:
infer_sampler

In [ ]:
from open3d.ml.datasets import utils

In [43]:
from open3d.ml.contrib import subsample

In [44]:
subsample?

In [45]:
model.preprocess(tile_old, {'split': 'test'})

{'point': array([[ 4.123    ,  4.5      , 20.796    ],
        [ 4.142    ,  4.4940033, 20.726    ],
        [ 4.0680003,  4.5350037, 20.711    ],
        ...,
        [ 2.0490003,  1.9360046, 19.813    ],
        [ 3.2040002,  3.459999 , 19.11     ],
        [ 3.1220002,  3.0270004, 19.178001 ]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32),
 'search_tree': <sklearn.neighbors._kd_tree.KDTree at 0x5eae403c8090>,
 'proj_inds': array([47971, 43877, 36730, ...,  7020, 53686,     0], dtype=int32)}

In [46]:
tile_old['point'].dtype

dtype('float64')

In [47]:
tile_old['point'] = tile_old['point'].astype(np.float32)

In [48]:
utils.DataProcessing.grid_subsampling(tile_old['point'].astype(np.float32), grid_size=0.02)

array([[-1.841, 96.335, 11.697],
       [-1.916, 96.421, 11.704],
       [-1.856, 96.338, 11.659],
       ...,
       [-4.051, 95.353, 11.818],
       [-2.195, 94.919,  9.81 ],
       [-2.214, 94.971,  9.837]], dtype=float32)

In [75]:
def get_cache(attr):
    return processed_data
infer_split = TorchDataloader(dataset=infer_dataset,
                            preprocess=model.preprocess,
                            transform=model.transform,
                            sampler=infer_sampler,
                            use_cache=False,                                    
                            cache_convert=get_cache)

In [76]:
model.transform??

In [77]:
batcher = DefaultBatcher()
infer_dataset = InferenceDummySplit(tile_old)

In [78]:
infer_loader = DataLoader(infer_split,
                            batch_size=12,
                            sampler=get_sampler(infer_sampler),
                            collate_fn=batcher.collate_fn)

In [79]:
len(infer_loader.dataset)

1

In [80]:
infer_loader.dataset[0]

now trying <class 'torch.Tensor'>


TypeError: min() received an invalid combination of arguments - got (out=NoneType, axis=int, ), but expected one of:
 * ()
 * (Tensor other)
 * (int dim, bool keepdim)
      didn't match because some of the keywords were incorrect: out, axis
 * (name dim, bool keepdim)
      didn't match because some of the keywords were incorrect: out, axis


In [67]:
data = next(iter(infer_loader))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
now trying <class 'torch.Tensor'>


KeyboardInterrupt: 

In [ ]:
model.preprocess(tile_old, {'split': 'test'})

{'point': array([[ 4.123    ,  4.5      , 20.796    ],
        [ 4.142    ,  4.4940033, 20.726    ],
        [ 4.0680003,  4.5350037, 20.711    ],
        ...,
        [ 2.0490003,  1.9360046, 19.813    ],
        [ 3.2040002,  3.459999 , 19.11     ],
        [ 3.1220002,  3.0270004, 19.178001 ]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32),
 'search_tree': <sklearn.neighbors._kd_tree.KDTree at 0x5fba14c7d0e0>,
 'proj_inds': array([47971, 43877, 36730, ...,  7020, 53686,     0], dtype=int32)}

In [ ]:
tile_old['label']

array([0, 0, 0, ..., 0, 0, 0], dtype=int32)

In [9]:
batch = next(iter(infer_loader))

NameError: name 'infer_loader' is not defined

In [ ]:
model.to(device)
model.device = device
model.eval()

preprocess_func = model.preprocess
processed_data = preprocess_func(data, {'split': 'test'})
def get_cache(attr):
    return processed_data

batcher = self.get_batcher(device)
infer_dataset = InferenceDummySplit(data)
self.dataset_split = infer_dataset
infer_sampler = infer_dataset.sampler
infer_split = TorchDataloader(dataset=infer_dataset,
                            preprocess=model.preprocess,
                            transform=model.transform,
                            sampler=infer_sampler,
                            use_cache=False,                                    
                            cache_convert=get_cache)
infer_loader = DataLoader(infer_split,
                            batch_size=cfg.batch_size,
                            sampler=get_sampler(infer_sampler),
                            collate_fn=batcher.collate_fn)

model.trans_point_sampler = infer_sampler.get_point_sampler()
self.curr_cloud_id = -1
self.test_probs = []
self.ori_test_probs = []
self.ori_test_labels = []

with torch.no_grad():
    for unused_step, inputs in enumerate(infer_loader):
        results = model(inputs['data'])
        self.update_tests(infer_sampler, inputs, results)

inference_result = {
    'predict_labels': self.ori_test_labels.pop(),
    'predict_scores': self.ori_test_probs.pop()
}

In [ ]:
pred = pipeline.run_inference(tile0)

now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>
now trying <class 'numpy.ndarray'>


TypeError: expected Tensor as element 0 in argument 0, but got NoneType

In [ ]:
%debug

> /Stor3/simone/semantic-segmentation/.pixi/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py(86)_wrapreduction()
     84                 return reduction(axis=axis, dtype=dtype, out=out, **passkwargs)
     85             else:
---> 86                 return reduction(axis=axis, out=out, **passkwargs)
     87 
     88     return ufunc.reduce(obj, axis, dtype, out, **passkwargs)

> /Stor3/simone/semantic-segmentation/.pixi/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py(2953)min()
   2951     6
   2952     """
-> 2953     return _wrapreduction(a, np.minimum, 'min', axis, None, out,
   2954                           keepdims=keepdims, initial=initial, where=where)
   2955 

> /home/simone/.local/lib/python3.12/site-packages/open3d/_ml3d/torch/models/point_transformer.py(297)transform()
    295                     points, labels = points[crop_idx], labels[crop_idx]
    296 
--> 297         points_min, points_max = np.min(points, 0), np.max(points, 0)


In [ ]:
tensor =  torch.ones([833437, 3])
tensor

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        ...,
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [ ]:
np.min(tensor, 0)

TypeError: min() received an invalid combination of arguments - got (out=NoneType, axis=int, ), but expected one of:
 * ()
 * (Tensor other)
 * (int dim, bool keepdim)
      didn't match because some of the keywords were incorrect: out, axis
 * (name dim, bool keepdim)
      didn't match because some of the keywords were incorrect: out, axis


In [ ]:
cfg = ml3d.utils.Config.load_from_file(cfg_path)

# Get data path, test folder and prediction folder from config file
data_path = cfg.dataset.get('dataset_path')
test_dir = cfg.dataset.get('test_dir')
test_pred_dir = cfg.dataset.get('test_result_folder', 'test_pred')

# Make prediction directory if it doesn't exist
if not os.path.exists(os.path.join(data_path, test_pred_dir)):
    os.mkdir(os.path.join(data_path, test_pred_dir))

    # Define model
    if cfg.model.name == 'RandLANet':
        model = CustomRandLANet(**cfg.model) 
    elif cfg.model.name == 'PointTransformer':
        model = CustomPointTransformer(**cfg.model)
    elif cfg.model.name == 'KPConv':
        model = CustomKPConv(**cfg.model)
    
    # Define pipeline
    pipeline = ml3d.pipelines.SemanticSegmentation(model, **cfg.pipeline)

    # Load models weights
    ckpt_path = cfg.model.ckpt_path
    pipeline.load_ckpt(ckpt_path)

    # Load data if given
    # if args.data is not None :
    #     # If specified as command line argument
    #     data = read_cloud(args.data)
    #     file_name = os.path.dirname(args.data)
    # elif cfg.dataset.get('file_name') is not None:
    #     # If specified in config file
    #     file_name = cfg.dataset.get('file_name')
    #     data = read_cloud(os.path.join(data_path, test_dir, file_name)) 
    # else:
    #     raise ValueError('No input data was specified.')

    # Store filenames of test point clouds in list
    if cfg.dataset.get('file_name') is not None:
        filenames = [cfg.dataset.get('file_name')]
    else:
        filenames = [f for f in os.listdir(os.path.join(data_path, test_dir)) if f[-3:] in FILE_TYPE]
    
    start = time.time()

    for filename in filenames:
        # Read point cloud
        print('reading', filename)
        data = read_cloud(os.path.join(data_path, test_dir, filename))

        # Run inference
        print('running inference for ', filename)
        pred = pipeline.run_inference(data)
        
        # Save input point cloud with prediction as txt file in 'prediction' folder
        print('saving result for ', filename)
        pcl_pred = np.hstack((data['point'], pred['predict_labels'].reshape(-1, 1)))
        # pcl_pred = np.hstack((data['point'], pred['predict_scores'][:, 1].reshape(-1, 1)))
        
        path_out = os.path.join(data_path, test_pred_dir, filename[:-3] + 'txt')
        np.savetxt(path_out, pcl_pred, fmt='%.3f')
    
    end = time.time()
    print("Inference took:", end-start, "seconds") 